In [15]:
%%capture
%pip install -qU langchain
%pip install -qU langchain-openai
%pip install python-dotenv



In [17]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAI
from langchain.prompts import (
    ChatPromptTemplate,
    FewShotPromptTemplate,
    PromptTemplate
)
from langchain.schema import HumanMessage, SystemMessage
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

In [18]:
load_dotenv()

True

In [19]:
import os
openAIKey = os.environ["OPENAI_API_KEY"]
print(f"OpenAI Key: {openAIKey[:10]}...{openAIKey[-10:]}")

OpenAI Key: sk-proj-5J...FleV4jNVUA


### Basic LLM

In [20]:
llm = OpenAI()
   
# Simple completion
response = llm.invoke("What are the benefits of Python programming in 100 words?")
print(f"Response: {response}\n")

Response: 

1. Easy to Learn and Use: Python has a simple and intuitive syntax, making it easy for beginners to learn and use.

2. Versatile and Flexible: Python can be used for a wide variety of applications, including web development, data analysis, artificial intelligence, and more.

3. Large and Active Community: Python has a massive and active community of developers, making it easy to find help and resources online.

4. Portable and Cross-platform: Python programs can run on multiple operating systems, making it a highly portable language.

5. Extensive Standard Library: Python comes with a large standard library that provides pre-written code to handle common tasks, saving time and effort for developers.

6. Integration and Compatibility: Python can easily integrate with other languages and tools, making it a popular choice for building complex software systems.

7. Increased Productivity: With its simple syntax and extensive libraries, Python allows developers to write code fas

In [14]:
### Chat model

In [7]:
chat = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")
   
# Create messages
messages = [
    SystemMessage(content="You are a helpful Python programming assistant."),
    HumanMessage(content="What's the difference between list and tuple in Python?")
]

response = chat.invoke(messages)
print(f"Chat Response: {response.content}\n")

Chat Response: In Python, lists and tuples are both used to store collections of items, but there are key differences between them:

1. Mutability:
- Lists are mutable, which means you can change, add, or remove elements after the list is created.
- Tuples are immutable, which means once a tuple is created, you cannot change its values.

2. Syntax:
- Lists are created using square brackets [].
- Tuples are created using parentheses ().

3. Performance:
- Tuples are generally faster than lists because they are immutable. This means that Python can optimize memory allocation for tuples.

4. Use case:
- Use lists when you need to work with a collection of items that may change over time.
- Use tuples when you want to ensure that the data in the collection cannot be modified accidentally.

Overall, the choice between using a list or a tuple depends on whether you need the ability to modify the collection after it has been created.



### Prompt Template

In [8]:
template = """You are an expert in {domain}.

Question: {question}

Please provide a detailed answer with examples."""

prompt = PromptTemplate(
input_variables=["domain", "question"],
template=template
)

# Format the prompt
formatted_prompt = prompt.format(
    domain="machine learning",
    question="What is gradient descent?"
)

# Use with LLM
llm = ChatOpenAI(temperature=0.5)
response = llm.invoke(formatted_prompt)
print(f"Template Response: {response.content}\n")

Template Response: Gradient descent is an optimization algorithm used in machine learning to minimize the loss function of a model by iteratively adjusting the parameters. The goal of gradient descent is to find the local minimum of the loss function by moving in the direction of steepest descent of the gradient.

The gradient descent algorithm works by taking small steps in the direction opposite to the gradient of the loss function with respect to the parameters. The size of the steps, known as the learning rate, determines how quickly the algorithm converges to the minimum. If the learning rate is too small, the algorithm may take a long time to converge, while if it is too large, the algorithm may overshoot the minimum.

There are different variants of gradient descent, including batch gradient descent, stochastic gradient descent, and mini-batch gradient descent. 

- Batch gradient descent calculates the gradient of the loss function with respect to all training examples and updat

### Few shot prompt

In [10]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "big", "output": "small"},
    {"input": "fast", "output": "slow"}
]

# Create example template
example_template = """
Input: {input}
Output: {output}
"""

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template=example_template
)

# Create few-shot prompt template
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Give the opposite of each input:",
    suffix="Input: {input}\nOutput:",
    input_variables=["input"]
)

# Use the prompt
llm = ChatOpenAI(temperature=0)
formatted = few_shot_prompt.format(input="cold")
print(f"Formatted Few-Shot Prompt:\n{formatted}\n")
response = llm.invoke(formatted)
print(f"Few-Shot Response: {response.content}\n")
   

Formatted Few-Shot Prompt:
Give the opposite of each input:


Input: happy
Output: sad



Input: big
Output: small



Input: fast
Output: slow


Input: cold
Output:

Few-Shot Response: hot



### Output parser

In [12]:
class MovieRecommendation(BaseModel):
    """Pydantic model for movie recommendations"""
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    genre: str = Field(description="Movie genre")
    rating: float = Field(description="Rating out of 10")
    reason: str = Field(description="Why this movie is recommended")
 
def output_parser_example():
    """Structured output parsing"""
    
    # Pydantic parser
    parser = PydanticOutputParser(pydantic_object=MovieRecommendation)
   
    # Create prompt with format instructions
    prompt = PromptTemplate(
        template="Recommend a movie about {topic}.\n{format_instructions}",
        input_variables=["topic"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
   
    # Generate and parse response
    llm = ChatOpenAI(temperature=0.7)
    formatted_prompt = prompt.format(topic="artificial intelligence")
    response = llm.invoke(formatted_prompt)
   
    try:
        parsed_output = parser.parse(response.content)
        print(f"Parsed Movie: {parsed_output}\n")
        print(f"Title: {parsed_output.title}")
        print(f"Year: {parsed_output.year}")
        print(f"Genre: {parsed_output.genre}")
        print(f"Rating: {parsed_output.rating}")
        print(f"Reason: {parsed_output.reason}\n")
    except Exception as e:
        print(f"Parsing error: {e}")

output_parser_example()


Parsed Movie: title='Ex Machina' year=2014 genre='Science Fiction' rating=8.0 reason='Ex Machina is a thought-provoking film that explores themes of artificial intelligence, consciousness, and ethics in a captivating way.'

Title: Ex Machina
Year: 2014
Genre: Science Fiction
Rating: 8.0
Reason: Ex Machina is a thought-provoking film that explores themes of artificial intelligence, consciousness, and ethics in a captivating way.



### Chat Prompt Template 

In [13]:
chat_template = ChatPromptTemplate.from_messages([
        ("system", "You are a {role} expert. Your tone should be {tone}."),
        ("user", "{user_input}"),
        ("assistant", "I'll help you with that. Let me think..."),
        ("user", "Please be specific and provide examples.")
    ])
   
# Format the messages
messages = chat_template.format_messages(
    role="Python programming",
    tone="friendly and educational",
    user_input="How do I handle exceptions in Python?"
)

# Send to chat model
chat = ChatOpenAI(temperature=0.5)
response = chat.invoke(messages)
print(f"Chat Template Response: {response.content}\n")

Chat Template Response: Of course! In Python, you can handle exceptions using try-except blocks. Here's an example:

```python
try:
    num1 = int(input("Enter a number: "))
    num2 = int(input("Enter another number: "))
    result = num1 / num2
    print("Result:", result)
except ZeroDivisionError:
    print("Error: Cannot divide by zero.")
except ValueError:
    print("Error: Please enter a valid number.")
except Exception as e:
    print("An error occurred:", e)
```

In this example:
- We try to get two numbers from the user and perform division.
- If a `ZeroDivisionError` occurs (dividing by zero), it will be caught and an error message will be displayed.
- If a `ValueError` occurs (invalid input), it will be caught and an error message will be displayed.
- If any other type of exception occurs, it will be caught by the generic `Exception` block, and the specific error message will be displayed.

You can customize the exception handling based on the specific types of errors you ex